# Практическая работа #6: Агрегация и объединение таблиц в SQL (Сквозная модель)
## Методический ориентир и эталонное решение (для преподавателя)

Блокнот содержит сквозное решение задач всех трех уровней сложности по единой предметной модели киберспортивного клуба: **все темы (агрегаты, GROUP BY, HAVING, INNER/LEFT JOIN, поиск сирот IS NULL, параметризация, экспорт CSV) представлены на каждом уровне**, но с возрастанием архитектурной сложности:

* **🟢 Easy (2 таблицы):** `members` + `game_sessions`. Базовые агрегаты по таблице, группировка по играм и городам, пороги через `HAVING`, `INNER JOIN` сессий с клиентами, поиск «спящих» клиентов через `LEFT JOIN + IS NULL`, подсчет суммарных трат гостей.
* **🟡 Medium (3 таблицы):** `hardware_pcs` + `members` + `game_sessions`. Полный конвейер `WHERE + GROUP BY + HAVING`, обязательная параметризация параметров через `?`, цепочка `JOIN` из 3 таблиц, выявление простаивающего оборудования, расчет эффективности игровых зон и видеокарт, экспорт витрины `zone_analytics_report.csv`.
* **🔴 Hard (3 таблицы):** Многомерная группировка (`GROUP BY zone_name, loyalty_tier`), динамический бенчмарк с подзапросом в `HAVING` (`> (SELECT AVG(...) ...)`), когортный анализ анонимов против держателей карт, эмуляция `FULL OUTER JOIN` в SQLite через `LEFT JOIN UNION ALL`, выгрузка витрины `executive_kpi_mart.csv`.
* **Критерии оценивания** на «5», «4», «3», «2».


In [1]:
import sqlite3
import pandas as pd
import pathlib

# Определяем базовый путь к папке работы
base_dir = pathlib.Path(".")
if not (base_dir / "easy").exists():
    base_dir = pathlib.Path(r"d:\GitHub\TexhProg-34\06 - Агрегация и объединение таблиц")

print("Рабочая директория:", base_dir.resolve())
print("Версия Pandas:", pd.__version__)
print("Версия SQLite3:", sqlite3.sqlite_version)


Рабочая директория: D:\GitHub\TexhProg-34\06 - Агрегация и объединение таблиц
Версия Pandas: 3.0.2
Версия SQLite3: 3.50.4


---
# 🟢 Часть 1. Эталонное решение Уровня Easy (Базовый)

### Задание 1.1–1.5: Подготовка базы данных и миграция CSV (вариант 1)


In [2]:
# Загрузка CSV-файлов варианта 1
easy_dir = base_dir / "easy" / "variant_01"
df_members_easy = pd.read_csv(easy_dir / "members.csv", sep=";")
df_sessions_easy = pd.read_csv(easy_dir / "game_sessions.csv", sep=";")

# Подключение к базе данных Easy
conn_easy = sqlite3.connect(":memory:") # Для чистоты демонстрации в памяти
conn_easy.execute("PRAGMA foreign_keys = ON;")
cur_easy = conn_easy.cursor()

# Создание структуры таблиц
cur_easy.execute('''
CREATE TABLE members (
    member_id INTEGER PRIMARY KEY,
    nickname TEXT NOT NULL,
    city TEXT NOT NULL,
    email TEXT,
    registered_at TEXT NOT NULL,
    loyalty_tier TEXT NOT NULL,
    balance INTEGER NOT NULL
);
''')

cur_easy.execute('''
CREATE TABLE game_sessions (
    session_id INTEGER PRIMARY KEY,
    member_id INTEGER REFERENCES members(member_id),
    game_title TEXT NOT NULL,
    session_date TEXT NOT NULL,
    hours_played REAL NOT NULL,
    discount_promo INTEGER NOT NULL,
    total_paid INTEGER NOT NULL,
    client_rating INTEGER
);
''')

# Импорт данных
df_members_easy.to_sql("members", conn_easy, if_exists="append", index=False)
df_sessions_easy.to_sql("game_sessions", conn_easy, if_exists="append", index=False)
conn_easy.commit()

# Аудит строк
audit_query = '''
SELECT 'members' AS tbl, COUNT(*) AS sql_count, ? AS pd_len FROM members
UNION ALL
SELECT 'game_sessions', COUNT(*), ? FROM game_sessions;
'''
pd.read_sql_query(audit_query, conn_easy, params=(len(df_members_easy), len(df_sessions_easy)))


tbl,sql_count,pd_len
members,100,100
game_sessions,250,250


### Задание 2. Базовые агрегаты по всей таблице
1. Общая статистика: число сессий, выручка, средняя длительность, мин/макс чек.
2. Пропущенные отзывы: `COUNT(*)` против `COUNT(client_rating)`.
3. Сессии с оценкой 5 звезд.


In [3]:
# 2.1 Общая статистика по сессиям клуба
q2_1 = '''
SELECT 
    COUNT(*) AS total_sessions,
    SUM(total_paid) AS total_revenue,
    ROUND(AVG(hours_played), 2) AS avg_hours,
    MIN(total_paid) AS min_ticket,
    MAX(total_paid) AS max_ticket
FROM game_sessions;
'''
pd.read_sql_query(q2_1, conn_easy)


total_sessions,total_revenue,avg_hours,min_ticket,max_ticket
250,132380,2.03,135,2000


In [4]:
# 2.2 Исследование пропущенных отзывов (NULL в агрегатах)
q2_2 = '''
SELECT 
    COUNT(*) AS total_sessions,
    COUNT(client_rating) AS sessions_with_rating,
    COUNT(*) - COUNT(client_rating) AS missing_ratings_count,
    ROUND(AVG(client_rating), 2) AS avg_rating_calculated
FROM game_sessions;
'''
pd.read_sql_query(q2_2, conn_easy)


total_sessions,sessions_with_rating,missing_ratings_count,avg_rating_calculated
250,193,57,3.99


In [5]:
# 2.3 Сессии с наивысшей оценкой 5 звезд
q2_3 = '''
SELECT 
    COUNT(*) AS five_star_sessions,
    ROUND(AVG(hours_played), 2) AS avg_hours_top_rated,
    SUM(total_paid) AS revenue_top_rated
FROM game_sessions
WHERE client_rating = 5;
'''
pd.read_sql_query(q2_3, conn_easy)


five_star_sessions,avg_hours_top_rated,revenue_top_rated
78,2.17,42963


### Задание 3. Группировка данных (`GROUP BY`)
1. Популярность игр (запуски, часы, средняя оценка).
2. География клиентов (число клиентов, депозиты).
3. Распределение по длительности сессий.


In [6]:
# 3.1 Популярность игр
q3_1 = '''
SELECT 
    game_title,
    COUNT(*) AS launches_count,
    ROUND(SUM(hours_played), 1) AS total_hours,
    ROUND(AVG(client_rating), 2) AS avg_rating
FROM game_sessions
GROUP BY game_title
ORDER BY launches_count DESC;
'''
pd.read_sql_query(q3_1, conn_easy)


game_title,launches_count,total_hours,avg_rating
Counter-Strike 2,47,89.0,3.92
Valorant,44,89.0,4.14
Apex Legends,35,67.0,3.75
PUBG,34,78.5,4.04
Cyberpunk 2077,34,71.0,3.96
GTA V,31,68.0,4.16
Dota 2,25,45.5,3.94


In [7]:
# 3.2 География клиентской базы
q3_2 = '''
SELECT 
    city,
    COUNT(*) AS members_count,
    SUM(balance) AS total_balance,
    ROUND(AVG(balance), 1) AS avg_balance
FROM members
GROUP BY city
ORDER BY total_balance DESC;
'''
pd.read_sql_query(q3_2, conn_easy)


city,members_count,total_balance,avg_balance
Москва,35,36950,1055.7
Санкт-Петербург,26,29000,1115.4
Казань,17,27100,1594.1
Новосибирск,15,23550,1570.0
Екатеринбург,7,5300,757.1


In [8]:
# 3.3 Распределение сессий по продолжительности
q3_3 = '''
SELECT 
    hours_played,
    COUNT(*) AS sessions_count,
    SUM(total_paid) AS revenue_by_duration
FROM game_sessions
GROUP BY hours_played
ORDER BY hours_played ASC;
'''
pd.read_sql_query(q3_3, conn_easy)


hours_played,sessions_count,revenue_by_duration
1.0,46,11365
1.5,64,24360
2.0,86,45970
3.0,32,25155
4.0,12,10080
5.0,10,15450


### Задание 4. Фильтрация вычисленных групп (`HAVING`)
1. Хиты клуба (более 25 запусков).
2. Высокооцененные игры (средний рейтинг >= 4.0).
3. Города с суммарным депозитом > 25 000 рублей.
4. Сквозной фильтр: сессии со скидкой (`discount_promo = 1`), сгруппированные по играм, с выручкой со скидками > 3 000 рублей.


In [9]:
# 4.1 Игры с количеством запусков более 25
q4_1 = '''
SELECT 
    game_title,
    COUNT(*) AS launches_count,
    SUM(total_paid) AS game_revenue
FROM game_sessions
GROUP BY game_title
HAVING COUNT(*) > 25
ORDER BY launches_count DESC;
'''
pd.read_sql_query(q4_1, conn_easy)


game_title,launches_count,game_revenue
Counter-Strike 2,47,23143
Valorant,44,22944
Apex Legends,35,17650
PUBG,34,20200
Cyberpunk 2077,34,19808
GTA V,31,17707


In [10]:
# 4.2 Игры со средней оценкой не ниже 4.0
q4_2 = '''
SELECT 
    game_title,
    COUNT(client_rating) AS rated_count,
    ROUND(AVG(client_rating), 2) AS avg_score
FROM game_sessions
GROUP BY game_title
HAVING AVG(client_rating) >= 4.0
ORDER BY avg_score DESC;
'''
pd.read_sql_query(q4_2, conn_easy)


game_title,rated_count,avg_score
GTA V,25,4.16
Valorant,35,4.14
PUBG,24,4.04


In [11]:
# 4.3 Города с суммарным депозитом клиентов более 25 000 руб.
q4_3 = '''
SELECT 
    city,
    COUNT(*) AS members_count,
    SUM(balance) AS total_city_balance
FROM members
GROUP BY city
HAVING SUM(balance) > 25000
ORDER BY total_city_balance DESC;
'''
pd.read_sql_query(q4_3, conn_easy)


city,members_count,total_city_balance
Москва,35,36950
Санкт-Петербург,26,29000
Казань,17,27100


In [12]:
# 4.4 Конвейер WHERE + GROUP BY + HAVING: сессии со скидкой, игры с выручкой > 3000
q4_4 = '''
SELECT 
    game_title,
    COUNT(*) AS promo_sessions,
    SUM(total_paid) AS promo_revenue
FROM game_sessions
WHERE discount_promo = 1
GROUP BY game_title
HAVING SUM(total_paid) > 3000
ORDER BY promo_revenue DESC;
'''
pd.read_sql_query(q4_4, conn_easy)


game_title,promo_sessions,promo_revenue
Valorant,16,7244
PUBG,6,4950
Dota 2,9,4028
GTA V,9,3532
Cyberpunk 2077,6,3308


### Задание 5. Соединения таблиц (`JOIN`)
1. `INNER JOIN`: сессии с именами клиентов (топ-10 по чеку).
2. `LEFT JOIN + IS NULL`: поиск «спящих» клиентов (0 сыгранных сессий).
3. `JOIN + GROUP BY`: суммарные траты каждого гостя.
4. `JOIN + GROUP BY + HAVING`: завсегдатаи с тратами > 3 500 рублей.


In [13]:
# 5.1 Сессии с именами клиентов (INNER JOIN)
q5_1 = '''
SELECT 
    s.session_id,
    m.nickname,
    m.city,
    s.game_title,
    s.hours_played,
    s.total_paid
FROM game_sessions s
JOIN members m ON s.member_id = m.member_id
ORDER BY s.total_paid DESC
LIMIT 10;
'''
pd.read_sql_query(q5_1, conn_easy)


session_id,nickname,city,game_title,hours_played,total_paid
10018,Lynx_299,Санкт-Петербург,PUBG,5.0,2000
10139,CyberKnight_918,Новосибирск,Cyberpunk 2077,5.0,2000
10235,Phantom_769,Екатеринбург,GTA V,5.0,2000
10156,Raven_169,Казань,PUBG,5.0,1800
10241,Reaper_58,Москва,Cyberpunk 2077,5.0,1800
10113,Rogue_689,Москва,GTA V,5.0,1500
10178,Wolf_555,Москва,Dota 2,5.0,1350
10187,Flash_648,Москва,Apex Legends,5.0,1250
10003,Frost_210,Казань,Counter-Strike 2,3.0,1200
10004,Lynx_253,Санкт-Петербург,GTA V,4.0,1200


In [14]:
# 5.2 Поиск спящих клиентов (LEFT JOIN + WHERE IS NULL)
q5_2 = '''
SELECT 
    m.member_id,
    m.nickname,
    m.city,
    m.loyalty_tier,
    m.balance
FROM members m
LEFT JOIN game_sessions s ON m.member_id = s.member_id
WHERE s.session_id IS NULL
ORDER BY m.balance DESC;
'''
df_sleeping = pd.read_sql_query(q5_2, conn_easy)
print(f"Найдено спящих клиентов без сессий: {len(df_sleeping)}")
df_sleeping.head(10)


Найдено спящих клиентов без сессий: 22


member_id,nickname,city,loyalty_tier,balance
88,Phoenix_797,Санкт-Петербург,Silver,4500
94,Hunter_748,Москва,Gold,4500
54,Grizzly_564,Санкт-Петербург,Gold,2500
86,Apex_866,Казань,Gold,2500
39,Titan_422,Москва,Silver,1500
40,Apex_121,Санкт-Петербург,Bronze,1500
60,Storm_206,Москва,Bronze,1500
96,Falcon_496,Казань,Silver,1500
98,Viper_978,Новосибирск,Platinum,1500
66,Grizzly_171,Санкт-Петербург,Bronze,1000


In [15]:
# 5.3 Индивидуальные траты гостей (JOIN + GROUP BY)
q5_3 = '''
SELECT 
    m.nickname,
    m.loyalty_tier,
    COUNT(s.session_id) AS sessions_played,
    SUM(s.total_paid) AS total_spent,
    ROUND(AVG(s.total_paid), 1) AS avg_ticket
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
GROUP BY m.member_id, m.nickname, m.loyalty_tier
ORDER BY total_spent DESC
LIMIT 10;
'''
pd.read_sql_query(q5_3, conn_easy)


nickname,loyalty_tier,sessions_played,total_spent,avg_ticket
Lynx_299,Bronze,5,4900,980.0
Flash_205,Silver,11,4477,407.0
Flash_648,Bronze,7,4340,620.0
Wolf_555,Bronze,9,4248,472.0
Rogue_689,Bronze,4,4200,1050.0
Lynx_253,Silver,6,3850,641.7
CyberKnight_918,Silver,4,3760,940.0
Kaiser_436,Bronze,7,3200,457.1
Cobra_406,Bronze,4,3070,767.5
Titan_323,Gold,7,2930,418.6


In [16]:
# 5.4 Завсегдатаи с тратами более 3 500 рублей (JOIN + GROUP BY + HAVING)
q5_4 = '''
SELECT 
    m.nickname,
    m.city,
    m.loyalty_tier,
    COUNT(s.session_id) AS sessions_played,
    SUM(s.total_paid) AS total_spent
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
GROUP BY m.member_id, m.nickname, m.city, m.loyalty_tier
HAVING SUM(s.total_paid) > 3500
ORDER BY total_spent DESC;
'''
pd.read_sql_query(q5_4, conn_easy)


nickname,city,loyalty_tier,sessions_played,total_spent
Lynx_299,Санкт-Петербург,Bronze,5,4900
Flash_205,Санкт-Петербург,Silver,11,4477
Flash_648,Москва,Bronze,7,4340
Wolf_555,Москва,Bronze,9,4248
Rogue_689,Москва,Bronze,4,4200
Lynx_253,Санкт-Петербург,Silver,6,3850
CyberKnight_918,Новосибирск,Silver,4,3760


In [17]:
# Завершение уровня Easy
conn_easy.close()
print("База Easy успешно закрыта.")


База Easy успешно закрыта.


---
# 🟡 Часть 2. Эталонное решение Уровня Medium (Средний)

### Задание 1. Перенос данных в SQL и аудит 3 таблиц (вариант 1)


In [18]:
# Загрузка CSV-файлов варианта 1 Medium
med_dir = base_dir / "medium" / "variant_01"
df_pcs_med = pd.read_csv(med_dir / "hardware_pcs.csv", sep=";")
df_members_med = pd.read_csv(med_dir / "members.csv", sep=";")
df_sessions_med = pd.read_csv(med_dir / "game_sessions.csv", sep=";")

# Подключение к базе данных Medium
conn_med = sqlite3.connect(":memory:")
conn_med.execute("PRAGMA foreign_keys = ON;")
cur_med = conn_med.cursor()

# Создание таблиц
cur_med.execute('''
CREATE TABLE hardware_pcs (
    pc_id INTEGER PRIMARY KEY,
    pc_number TEXT NOT NULL,
    zone_name TEXT NOT NULL,
    hourly_rate INTEGER NOT NULL,
    gpu TEXT NOT NULL,
    cpu TEXT NOT NULL,
    ram_gb INTEGER NOT NULL,
    status TEXT NOT NULL
);
''')

cur_med.execute('''
CREATE TABLE members (
    member_id INTEGER PRIMARY KEY,
    nickname TEXT NOT NULL,
    city TEXT NOT NULL,
    email TEXT,
    registered_at TEXT NOT NULL,
    loyalty_tier TEXT NOT NULL,
    balance INTEGER NOT NULL
);
''')

cur_med.execute('''
CREATE TABLE game_sessions (
    session_id INTEGER PRIMARY KEY,
    member_id INTEGER REFERENCES members(member_id),
    pc_id INTEGER NOT NULL REFERENCES hardware_pcs(pc_id),
    game_title TEXT NOT NULL,
    session_date TEXT NOT NULL,
    hours_played REAL NOT NULL,
    discount_promo INTEGER NOT NULL,
    total_paid INTEGER NOT NULL,
    client_rating INTEGER
);
''')

# Импорт данных
df_pcs_med.to_sql("hardware_pcs", conn_med, if_exists="append", index=False)
df_members_med.to_sql("members", conn_med, if_exists="append", index=False)
df_sessions_med.to_sql("game_sessions", conn_med, if_exists="append", index=False)
conn_med.commit()

# Аудит строк
audit_med_q = '''
SELECT 'hardware_pcs' AS tbl, COUNT(*) AS sql_count, ? AS pd_len FROM hardware_pcs
UNION ALL
SELECT 'members', COUNT(*), ? FROM members
UNION ALL
SELECT 'game_sessions', COUNT(*), ? FROM game_sessions;
'''
pd.read_sql_query(audit_med_q, conn_med, params=(len(df_pcs_med), len(df_members_med), len(df_sessions_med)))


tbl,sql_count,pd_len
hardware_pcs,30,30
members,150,150
game_sessions,500,500


### Задание 2. Агрегатный аудит и анализ пропусков (Medium)
1. Экономика клуба одной строкой.
2. Анализ анонимного потока (`member_id IS NULL`).
3. Аудит активности обратной связи (процент выставленных оценок).


In [19]:
# 2.1 Экономика клуба одной строкой
q_med_2_1 = '''
SELECT 
    COUNT(*) AS total_sessions,
    SUM(total_paid) AS total_revenue,
    ROUND(AVG(total_paid)) AS avg_check,
    ROUND(AVG(client_rating), 2) AS avg_rating,
    ROUND(SUM(hours_played), 1) AS total_hours
FROM game_sessions;
'''
pd.read_sql_query(q_med_2_1, conn_med)


total_sessions,total_revenue,avg_check,avg_rating,total_hours
500,233283,467.0,3.97,1060.0


In [20]:
# 2.2 Анализ анонимного потока (member_id IS NULL)
q_med_2_2 = '''
SELECT 
    CASE WHEN member_id IS NULL THEN 'Анонимный гость' ELSE 'Постоянный клиент' END AS client_type,
    COUNT(*) AS sessions_count,
    SUM(total_paid) AS revenue,
    ROUND(AVG(total_paid), 1) AS avg_check,
    ROUND(AVG(hours_played), 2) AS avg_duration
FROM game_sessions
GROUP BY (member_id IS NULL);
'''
pd.read_sql_query(q_med_2_2, conn_med)


client_type,sessions_count,revenue,avg_check,avg_duration
Постоянный клиент,445,212248,477.0,2.13
Анонимный гость,55,21035,382.5,2.01


In [21]:
# 2.3 Аудит активности обратной связи
q_med_2_3 = '''
SELECT 
    COUNT(*) AS total_sessions,
    COUNT(client_rating) AS reviews_left,
    COUNT(*) - COUNT(client_rating) AS reviews_omitted,
    ROUND(100.0 * COUNT(client_rating) / COUNT(*), 1) AS feedback_share_percent
FROM game_sessions;
'''
pd.read_sql_query(q_med_2_3, conn_med)


total_sessions,reviews_left,reviews_omitted,feedback_share_percent
500,402,98,80.4


### Задание 3. Группировки и составной конвейер (Medium)
1. Рейтинг дисциплин: сессии > 35, сортировка по выручке.
2. Финансовая состоятельность городов: средний баланс >= 1000 рублей.
3. Сквозной конвейер: сессии без скидки (`discount_promo = 0`), длительностью >= 2 часов, средний чек > 500 рублей с параметризацией `?`.


In [22]:
# 3.1 Рейтинг игровых дисциплин (сессий > 35)
q_med_3_1 = '''
SELECT 
    game_title,
    COUNT(*) AS sessions_count,
    SUM(total_paid) AS revenue,
    ROUND(AVG(client_rating), 2) AS avg_rating
FROM game_sessions
GROUP BY game_title
HAVING COUNT(*) > 35
ORDER BY revenue DESC;
'''
pd.read_sql_query(q_med_3_1, conn_med)


game_title,sessions_count,revenue,avg_rating
Atomic Heart,65,30978,3.71
Cyberpunk 2077,51,27888,4.26
Counter-Strike 2,55,25928,4.13
Valorant,50,25079,3.65
Apex Legends,50,24742,4.05
Rust,49,22109,4.16
PUBG,53,22027,3.67
Warzone,48,20698,3.95
Dota 2,43,19922,4.13
GTA V,36,13912,4.10


In [23]:
# 3.2 Финансовая состоятельность городов (средний баланс >= 1000 руб.)
q_med_3_2 = '''
SELECT 
    city,
    COUNT(*) AS members_count,
    ROUND(AVG(balance), 1) AS avg_balance,
    MAX(balance) AS max_balance
FROM members
GROUP BY city
HAVING AVG(balance) >= 1000
ORDER BY avg_balance DESC;
'''
pd.read_sql_query(q_med_3_2, conn_med)


city,members_count,avg_balance,max_balance
Санкт-Петербург,26,2723.1,6000
Казань,24,2131.3,6000
Новосибирск,16,1781.3,6000
Москва,71,1574.6,6000
Екатеринбург,13,1115.4,6000


In [24]:
# 3.3 Сквозной конвейер с параметризацией (?)
q_med_3_3 = '''
SELECT 
    game_title,
    COUNT(*) AS qualifying_sessions,
    ROUND(AVG(total_paid), 1) AS avg_ticket,
    SUM(total_paid) AS group_revenue
FROM game_sessions
WHERE discount_promo = ? AND hours_played >= ?
GROUP BY game_title
HAVING AVG(total_paid) > ?
ORDER BY group_revenue DESC;
'''
# Передаем параметры: discount_promo = 0, hours_played >= 2.0, avg_ticket > 500
pd.read_sql_query(q_med_3_3, conn_med, params=(0, 2.0, 500))


game_title,qualifying_sessions,avg_ticket,group_revenue
Atomic Heart,42,516.3,21685
Cyberpunk 2077,26,691.0,17965
Valorant,29,605.0,17545
Apex Legends,30,566.5,16995
Rust,27,571.7,15435
Counter-Strike 2,24,614.6,14750
Warzone,23,567.0,13040
Dota 2,20,565.3,11305


### Задание 4. Соединения таблиц (Medium)
1. Сквозной журнал (цепочка 3 таблиц: `game_sessions` + `members` + `hardware_pcs`).
2. Поиск простаивающего оборудования (`hardware_pcs LEFT JOIN ... WHERE session_id IS NULL`).
3. Поиск спящей клиентской базы.


In [25]:
# 4.1 Сквозной журнал (цепочка 3 таблиц)
q_med_4_1 = '''
SELECT 
    s.session_id,
    m.nickname,
    p.pc_number,
    p.zone_name,
    s.game_title,
    s.total_paid
FROM game_sessions s
JOIN members m ON s.member_id = m.member_id
JOIN hardware_pcs p ON s.pc_id = p.pc_id
ORDER BY s.total_paid DESC
LIMIT 10;
'''
pd.read_sql_query(q_med_4_1, conn_med)


session_id,nickname,pc_number,zone_name,game_title,total_paid
20320,Hunter_845,PC-26,VIP,Dota 2,2520
20491,Ronin_809,PC-26,VIP,Valorant,2520
20380,Lynx_415,PC-23,VIP,Cyberpunk 2077,2400
20324,Phantom_631,PC-26,VIP,Cyberpunk 2077,1680
20117,Ghost_658,PC-22,Bootcamp,Warzone,1620
20127,Shadow_140,PC-25,VIP,Valorant,1600
20170,Ronin_809,PC-23,VIP,Warzone,1200
20189,Ronin_908,PC-23,VIP,Dota 2,1200
20192,Flash_363,PC-25,VIP,Rust,1200
20304,Lynx_898,PC-23,VIP,Counter-Strike 2,1200


In [26]:
# 4.2 Поиск простаивающих ПК (LEFT JOIN + WHERE IS NULL)
q_med_4_2 = '''
SELECT 
    p.pc_id,
    p.pc_number,
    p.zone_name,
    p.gpu,
    p.status
FROM hardware_pcs p
LEFT JOIN game_sessions s ON p.pc_id = s.pc_id
WHERE s.session_id IS NULL;
'''
df_idle_pcs = pd.read_sql_query(q_med_4_2, conn_med)
print(f"Найдено простаивающих ПК: {len(df_idle_pcs)}")
df_idle_pcs


Найдено простаивающих ПК: 3


pc_id,pc_number,zone_name,gpu,status
28,PC-28,PS5,PlayStation 5 Custom GPU,Maintenance
29,PC-29,PS5,PlayStation 5 Custom GPU,Reserved
30,PC-30,PS5,PlayStation 5 Custom GPU,Reserved


In [27]:
# 4.3 Поиск спящих клиентов
q_med_4_3 = '''
SELECT 
    m.member_id,
    m.nickname,
    m.city,
    m.registered_at,
    m.balance
FROM members m
LEFT JOIN game_sessions s ON m.member_id = s.member_id
WHERE s.session_id IS NULL
ORDER BY m.balance DESC;
'''
df_sleeping_med = pd.read_sql_query(q_med_4_3, conn_med)
print(f"Найдено спящих клиентов: {len(df_sleeping_med)}")
df_sleeping_med.head(10)


Найдено спящих клиентов: 26


member_id,nickname,city,registered_at,balance
92,Matrix_5926,Санкт-Петербург,2025-11-12,6000
133,Titan_155,Москва,2025-05-25,6000
134,Grizzly_181,Санкт-Петербург,2025-10-09,6000
82,Falcon_886,Москва,2025-03-26,2000
138,Grizzly_963,Санкт-Петербург,2025-12-09,2000
139,Ronin_223,Москва,2025-07-11,2000
141,Viper_518,Москва,2025-03-15,2000
144,Turbo_480,Москва,2025-04-04,2000
145,Cobra_921,Казань,2026-01-16,2000
146,Ronin_49,Санкт-Петербург,2025-06-14,2000


### Задание 5. Бизнес-аналитика на связках (Medium)
1. Экономика игровых зон (`hardware_pcs` + `game_sessions`).
2. Загрузка видеокарт (`gpu`).
3. Параметризованная выборка лояльных гостей (сессий >= 4, выручка >= 2500 руб.).


In [28]:
# 5.1 Экономическая эффективность игровых зон
q_med_5_1 = '''
SELECT 
    p.zone_name,
    COUNT(DISTINCT p.pc_id) AS total_pcs,
    COUNT(s.session_id) AS total_sessions,
    ROUND(SUM(s.hours_played), 1) AS hours_billed,
    SUM(s.total_paid) AS total_revenue,
    ROUND(AVG(s.total_paid), 1) AS avg_ticket
FROM hardware_pcs p
JOIN game_sessions s ON p.pc_id = s.pc_id
GROUP BY p.zone_name
ORDER BY total_revenue DESC;
'''
df_zone_analytics = pd.read_sql_query(q_med_5_1, conn_med)
df_zone_analytics


zone_name,total_pcs,total_sessions,hours_billed,total_revenue,avg_ticket
Standard,14,265,573.0,86600,326.8
Bootcamp,8,145,308.0,75523,520.8
VIP,5,90,179.0,71160,790.7


In [29]:
# 5.2 Загрузка видеокарт
q_med_5_2 = '''
SELECT 
    p.gpu,
    COUNT(DISTINCT p.pc_id) AS pcs_with_gpu,
    ROUND(SUM(s.hours_played), 1) AS hours_played,
    SUM(s.total_paid) AS revenue_generated
FROM hardware_pcs p
JOIN game_sessions s ON p.pc_id = s.pc_id
GROUP BY p.gpu
ORDER BY revenue_generated DESC;
'''
pd.read_sql_query(q_med_5_2, conn_med)


gpu,pcs_with_gpu,hours_played,revenue_generated
RTX 4060 Ti,5,202.5,50395
RTX 2060,6,255.5,36531
GTX 1660 Super,5,205.0,31978
RTX 4070 Ti,2,72.0,29220
RTX 4090,2,74.0,28860
RTX 4060,3,105.5,25128
RTX 3050,3,112.5,18091
RTX 4080 Super,1,33.0,13080


In [30]:
# 5.3 Целевая аудитория клуба (параметризовано: 4 сессии, 2500 руб.)
q_med_5_3 = '''
SELECT 
    m.nickname,
    m.city,
    m.loyalty_tier,
    COUNT(s.session_id) AS sessions_played,
    SUM(s.total_paid) AS customer_ltv
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
GROUP BY m.member_id, m.nickname, m.city, m.loyalty_tier
HAVING COUNT(s.session_id) >= ? AND SUM(s.total_paid) >= ?
ORDER BY customer_ltv DESC;
'''
pd.read_sql_query(q_med_5_3, conn_med, params=(4, 2500))


nickname,city,loyalty_tier,sessions_played,customer_ltv
Lynx_415,Екатеринбург,Silver,8,5885
Ronin_809,Москва,Bronze,6,5368
Falcon_548,Москва,Silver,7,4830
Phantom_730,Новосибирск,Gold,8,4170
Phantom_631,Москва,Bronze,5,3860
Flash_338,Москва,Bronze,6,3830
Shadow_140,Москва,Bronze,5,3780
Lynx_898,Екатеринбург,Gold,5,3740
Hunter_845,Москва,Gold,4,3480
Ghost_658,Москва,Bronze,4,3165


### Задание 6. Экспорт аналитических срезов (Medium)


In [31]:
# Экспорт отчета по зонам в CSV
out_csv_med = base_dir / "zone_analytics_report.csv"
df_zone_analytics.to_csv(out_csv_med, sep=";", index=False, encoding="utf-8-sig")
print(f"Отчет экспортирован в {out_csv_med.name}")

conn_med.close()
print("База Medium закрыта.")


Отчет экспортирован в zone_analytics_report.csv
База Medium закрыта.


---
# 🔴 Часть 3. Эталонное решение Уровня Hard (Продвинутый)

### Задание 1. Архитектура схемы и импорт 3 таблиц (вариант 1 Hard)


In [32]:
# Загрузка CSV-файлов варианта 1 Hard
hard_dir = base_dir / "hard" / "variant_01"
df_pcs_hard = pd.read_csv(hard_dir / "hardware_pcs.csv", sep=";")
df_members_hard = pd.read_csv(hard_dir / "members.csv", sep=";")
df_sessions_hard = pd.read_csv(hard_dir / "game_sessions.csv", sep=";")

# Подключение к базе данных Hard
conn_hard = sqlite3.connect(":memory:")
conn_hard.execute("PRAGMA foreign_keys = ON;")
cur_hard = conn_hard.cursor()

# Создание таблиц
cur_hard.execute('''
CREATE TABLE hardware_pcs (
    pc_id INTEGER PRIMARY KEY,
    pc_number TEXT NOT NULL,
    zone_name TEXT NOT NULL,
    hourly_rate INTEGER NOT NULL,
    gpu TEXT NOT NULL,
    cpu TEXT NOT NULL,
    ram_gb INTEGER NOT NULL,
    status TEXT NOT NULL
);
''')

cur_hard.execute('''
CREATE TABLE members (
    member_id INTEGER PRIMARY KEY,
    nickname TEXT NOT NULL,
    city TEXT NOT NULL,
    email TEXT,
    registered_at TEXT NOT NULL,
    loyalty_tier TEXT NOT NULL,
    balance INTEGER NOT NULL
);
''')

cur_hard.execute('''
CREATE TABLE game_sessions (
    session_id INTEGER PRIMARY KEY,
    member_id INTEGER REFERENCES members(member_id),
    pc_id INTEGER NOT NULL REFERENCES hardware_pcs(pc_id),
    game_title TEXT NOT NULL,
    session_date TEXT NOT NULL,
    hours_played REAL NOT NULL,
    discount_promo INTEGER NOT NULL,
    total_paid INTEGER NOT NULL,
    client_rating INTEGER
);
''')

# Импорт данных
df_pcs_hard.to_sql("hardware_pcs", conn_hard, if_exists="append", index=False)
df_members_hard.to_sql("members", conn_hard, if_exists="append", index=False)
df_sessions_hard.to_sql("game_sessions", conn_hard, if_exists="append", index=False)
conn_hard.commit()

# Аудит строк
audit_hard_q = '''
SELECT 'hardware_pcs' AS tbl, COUNT(*) AS sql_count, ? AS pd_len FROM hardware_pcs
UNION ALL
SELECT 'members', COUNT(*), ? FROM members
UNION ALL
SELECT 'game_sessions', COUNT(*), ? FROM game_sessions;
'''
pd.read_sql_query(audit_hard_q, conn_hard, params=(len(df_pcs_hard), len(df_members_hard), len(df_sessions_hard)))


tbl,sql_count,pd_len
hardware_pcs,40,40
members,200,200
game_sessions,800,800


### Задание 2. Продвинутые агрегаты и когортный аудит (Hard)
1. Сводный дашборд показателей клуба (уникальные клиенты, задействованные ПК).
2. Сравнительный анализ: зарегистрированные гости против анонимов.


In [33]:
# 2.1 Сводный дашборд показателей клуба
q_hard_2_1 = '''
SELECT 
    COUNT(*) AS total_sessions,
    SUM(total_paid) AS total_revenue,
    ROUND(AVG(total_paid), 1) AS avg_check,
    COUNT(DISTINCT member_id) AS unique_players_served,
    COUNT(DISTINCT pc_id) AS active_pcs_utilized,
    ROUND(AVG(hours_played), 2) AS avg_session_hours,
    ROUND(AVG(client_rating), 2) AS customer_csat
FROM game_sessions;
'''
pd.read_sql_query(q_hard_2_1, conn_hard)


total_sessions,total_revenue,avg_check,unique_players_served,active_pcs_utilized,avg_session_hours,customer_csat
800,447606,559.5,167,37,2.31,4.07


In [34]:
# 2.2 Сравнительный анализ: клубные гости vs анонимы
q_hard_2_2 = '''
SELECT 
    CASE WHEN member_id IS NULL THEN 'Анонимный гость' ELSE 'Клубный клиент' END AS segment,
    COUNT(*) AS sessions_count,
    SUM(total_paid) AS segment_revenue,
    ROUND(100.0 * SUM(total_paid) / (SELECT SUM(total_paid) FROM game_sessions), 1) AS revenue_share_pct,
    ROUND(AVG(total_paid), 1) AS avg_check,
    ROUND(AVG(hours_played), 2) AS avg_duration,
    ROUND(SUM(total_paid) / SUM(hours_played), 1) AS revenue_per_hour
FROM game_sessions
GROUP BY (member_id IS NULL);
'''
pd.read_sql_query(q_hard_2_2, conn_hard)


segment,sessions_count,segment_revenue,revenue_share_pct,avg_check,avg_duration,revenue_per_hour
Клубный клиент,723,408496,91.3,565.0,2.31,244.2
Анонимный гость,77,39110,8.7,507.9,2.31,219.7


### Задание 3. Многомерная группировка и динамический бенчмарк в HAVING (Hard)
1. Многомерный срез (Зона × Уровень лояльности).
2. Динамический бенчмарк: игры с чеком строго выше среднего чека клуба через подзапрос.
3. Экономика рангов лояльности (LTV/ARPU).


In [35]:
# 3.1 Многомерный срез (Зона × Уровень лояльности)
q_hard_3_1 = '''
SELECT 
    p.zone_name,
    m.loyalty_tier,
    COUNT(s.session_id) AS sessions_count,
    SUM(s.total_paid) AS segment_revenue,
    ROUND(AVG(s.total_paid), 1) AS avg_ticket
FROM game_sessions s
JOIN hardware_pcs p ON s.pc_id = p.pc_id
JOIN members m ON s.member_id = m.member_id
GROUP BY p.zone_name, m.loyalty_tier
HAVING COUNT(s.session_id) >= 3
ORDER BY segment_revenue DESC;
'''
df_executive_mart = pd.read_sql_query(q_hard_3_1, conn_hard)
df_executive_mart.head(15)


zone_name,loyalty_tier,sessions_count,segment_revenue,avg_ticket
Bootcamp,Bronze,87,50007,574.8
VIP,Bronze,48,49842,1038.4
VIP,Silver,41,42352,1033.0
VIP,Gold,35,38170,1090.6
Standard,Gold,107,38128,356.3
Standard,Bronze,108,37805,350.0
Standard,Silver,97,35331,364.2
Bootcamp,Gold,58,33617,579.6
Bootcamp,Silver,44,25296,574.9
VIP,Platinum,19,22640,1191.6


In [36]:
# 3.2 Динамический бенчмарк (подзапрос в HAVING)
q_hard_3_2 = '''
SELECT 
    game_title,
    COUNT(*) AS sessions_count,
    ROUND(AVG(total_paid), 1) AS avg_game_ticket,
    ROUND((SELECT AVG(total_paid) FROM game_sessions), 1) AS club_benchmark_avg
FROM game_sessions
GROUP BY game_title
HAVING AVG(total_paid) > (SELECT AVG(total_paid) FROM game_sessions)
ORDER BY avg_game_ticket DESC;
'''
pd.read_sql_query(q_hard_3_2, conn_hard)


game_title,sessions_count,avg_game_ticket,club_benchmark_avg
Apex Legends,72,611.3,559.5
Cyberpunk 2077,70,610.7,559.5
Dota 2,87,594.4,559.5
Warzone,75,572.1,559.5


In [37]:
# 3.3 Экономика рангов лояльности (ARPU по рангам)
q_hard_3_3 = '''
SELECT 
    m.loyalty_tier,
    COUNT(DISTINCT m.member_id) AS total_members_in_tier,
    COUNT(s.session_id) AS sessions_played,
    SUM(s.total_paid) AS tier_revenue,
    ROUND(1.0 * SUM(s.total_paid) / COUNT(DISTINCT m.member_id), 1) AS arpu_per_member
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
GROUP BY m.loyalty_tier
ORDER BY arpu_per_member DESC;
'''
pd.read_sql_query(q_hard_3_3, conn_hard)


loyalty_tier,total_members_in_tier,sessions_played,tier_revenue,arpu_per_member
Platinum,21,98,57948,2759.4
Silver,41,182,102979,2511.7
Bronze,58,243,137654,2373.3
Gold,47,200,109915,2338.6


### Задание 4. Сложные соединения и эмуляция FULL OUTER JOIN (Hard)
1. Аудит премиальных сессий (VIP / Bootcamp, длительность >= 3 ч, оценка 5).
2. Двусторонний поиск сирот (ПК без сессий и клиенты без сессий).
3. Эмуляция `FULL OUTER JOIN` в SQLite.


In [38]:
# 4.1 Аудит премиальных сессий
q_hard_4_1 = '''
SELECT 
    s.session_id,
    m.nickname,
    m.city,
    p.pc_number,
    p.zone_name,
    s.game_title,
    s.hours_played,
    s.total_paid,
    s.client_rating
FROM game_sessions s
JOIN members m ON s.member_id = m.member_id
JOIN hardware_pcs p ON s.pc_id = p.pc_id
WHERE p.zone_name IN ('VIP', 'Bootcamp')
  AND s.hours_played >= 3.0
  AND s.client_rating = 5
ORDER BY s.total_paid DESC
LIMIT 10;
'''
pd.read_sql_query(q_hard_4_1, conn_hard)


session_id,nickname,city,pc_number,zone_name,game_title,hours_played,total_paid,client_rating
30500,Flash_533,Казань,PC-36,VIP,Dota 2,6.0,2520,5
30517,Samurai_220,Казань,PC-31,VIP,Valorant,5.0,2200,5
30094,Samurai_220,Казань,PC-37,VIP,Cyberpunk 2077,5.0,2000,5
30109,Reaper_915,Екатеринбург,PC-32,VIP,Warzone,5.0,2000,5
30352,Neo_832,Казань,PC-37,VIP,Dota 2,5.0,1800,5
30286,Flash_533,Казань,PC-33,VIP,Apex Legends,4.0,1760,5
30617,Echo_830,Москва,PC-35,VIP,Rust,4.0,1760,5
30028,CyberKnight_672,Казань,PC-30,Bootcamp,Cyberpunk 2077,6.0,1680,5
30118,Shadow_637,Москва,PC-37,VIP,Apex Legends,4.0,1600,5
30482,Rogue_881,Екатеринбург,PC-37,VIP,Counter-Strike 2,4.0,1600,5


In [39]:
# 4.2 Двусторонний поиск несвязанных элементов
# 4.2.1 Компьютеры без сессий
q_hard_4_2_pcs = '''
SELECT 'Простаивающий ПК' AS entity_type, pc_number AS identifier, zone_name AS details
FROM hardware_pcs p
LEFT JOIN game_sessions s ON p.pc_id = s.pc_id
WHERE s.session_id IS NULL;
'''
# 4.2.2 Клиенты без сессий
q_hard_4_2_members = '''
SELECT 'Спящий клиент' AS entity_type, nickname AS identifier, city || ' (' || loyalty_tier || ')' AS details
FROM members m
LEFT JOIN game_sessions s ON m.member_id = s.member_id
WHERE s.session_id IS NULL;
'''
df_idle_all = pd.concat([
    pd.read_sql_query(q_hard_4_2_pcs, conn_hard),
    pd.read_sql_query(q_hard_4_2_members, conn_hard)
], ignore_index=True)
print(f"Всего несвязанных сущностей: {len(df_idle_all)}")
df_idle_all.head(15)


Всего несвязанных сущностей: 36


entity_type,identifier,details
Простаивающий ПК,PC-38,PS5
Простаивающий ПК,PC-39,PS5
Простаивающий ПК,PC-40,PS5
Спящий клиент,Lynx_503,Санкт-Петербург (Bronze)
Спящий клиент,Reaper_268,Санкт-Петербург (Platinum)
Спящий клиент,Neo_186,Санкт-Петербург (Silver)
Спящий клиент,Falcon_253,Казань (Silver)
Спящий клиент,Flash_640,Санкт-Петербург (Bronze)
Спящий клиент,Reaper_232,Санкт-Петербург (Silver)
Спящий клиент,Grizzly_672,Москва (Bronze)


In [40]:
# 4.3 Эмуляция FULL OUTER JOIN в SQLite
# Совмещаем клиентов и сессии:
# 1) Все клиенты со своими сессиями + клиенты без сессий (LEFT JOIN)
# 2) Анонимные сессии без клиентов (где member_id IS NULL)
q_hard_4_3_full = '''
-- Ветка 1: Все клиенты (с сессиями и без)
SELECT 
    m.member_id,
    m.nickname,
    s.session_id,
    s.game_title,
    s.total_paid,
    'Клиент с сессией / без сессии' AS record_origin
FROM members m
LEFT JOIN game_sessions s ON m.member_id = s.member_id

UNION ALL

-- Ветка 2: Сессии анонимов (нет в первой ветке)
SELECT 
    NULL AS member_id,
    '<< Анонимный гость >>' AS nickname,
    s.session_id,
    s.game_title,
    s.total_paid,
    'Анонимная сессия' AS record_origin
FROM game_sessions s
WHERE s.member_id IS NULL;
'''
df_full_emulated = pd.read_sql_query(q_hard_4_3_full, conn_hard)
print("Всего строк в эмулированном FULL JOIN:", len(df_full_emulated))
print("Распределение по происхождению записей:")
print(df_full_emulated['record_origin'].value_counts())


Всего строк в эмулированном FULL JOIN: 833
Распределение по происхождению записей:
record_origin
Клиент с сессией / без сессии    756
Анонимная сессия                  77
Name: count, dtype: int64


### Задание 5. Эффективность оборудования и параметризованная выборка (Hard)
1. Эффективность парка ПК по зонам (машино-часы и удельная выручка на машину).
2. Таргетированная выборка хайроллеров через параметры `?`.


In [41]:
# 5.1 Эффективность парка оборудования по зонам
q_hard_5_1 = '''
SELECT 
    p.zone_name,
    COUNT(DISTINCT p.pc_id) AS pcs_in_zone,
    ROUND(SUM(s.hours_played), 1) AS total_machine_hours,
    SUM(s.total_paid) AS zone_revenue,
    ROUND(1.0 * SUM(s.total_paid) / COUNT(DISTINCT p.pc_id), 1) AS revenue_per_pc
FROM hardware_pcs p
JOIN game_sessions s ON p.pc_id = s.pc_id
GROUP BY p.zone_name
ORDER BY revenue_per_pc DESC;
'''
pd.read_sql_query(q_hard_5_1, conn_hard)


zone_name,pcs_in_zone,total_machine_hours,zone_revenue,revenue_per_pc
VIP,7,387.0,160584,22940.6
Bootcamp,12,553.5,143558,11963.2
Standard,18,910.0,143464,7970.2


In [42]:
# 5.2 Таргетированная выборка хайроллеров (параметризовано через ?)
q_hard_5_2 = '''
SELECT 
    m.nickname,
    m.city,
    m.loyalty_tier,
    COUNT(s.session_id) AS visits_count,
    SUM(s.total_paid) AS total_spend
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
WHERE m.city IN (?, ?)
GROUP BY m.member_id, m.nickname, m.city, m.loyalty_tier
HAVING COUNT(s.session_id) >= ? AND SUM(s.total_paid) > ?
ORDER BY total_spend DESC;
'''
pd.read_sql_query(q_hard_5_2, conn_hard, params=('Москва', 'Санкт-Петербург', 3, 3000))


nickname,city,loyalty_tier,visits_count,total_spend
Wolf_358,Москва,Platinum,11,6950
Grizzly_542,Москва,Silver,11,6390
Reaper_351,Москва,Gold,8,6134
Blaze_382,Москва,Platinum,7,5420
Sniper_278,Москва,Bronze,8,5364
CyberKnight_975,Москва,Bronze,11,5204
Cobra_648,Москва,Bronze,8,4856
Phantom_98,Москва,Bronze,7,4803
Sniper_293,Москва,Gold,6,4765
Echo_830,Москва,Bronze,6,4680


### Задание 6. Экспорт аналитической витрины (Hard)


In [43]:
# Экспорт управленческой витрины Зона x Ранг в CSV
out_csv_hard = base_dir / "executive_kpi_mart.csv"
df_executive_mart.to_csv(out_csv_hard, sep=";", index=False, encoding="utf-8-sig")
print(f"Витрина экспортирована в {out_csv_hard.name}")

conn_hard.close()
print("База Hard закрыта.")


Витрина экспортирована в executive_kpi_mart.csv
База Hard закрыта.


---
# 📊 Критерии оценивания практической работы #6

| Оценка | Требования к выполнению |
|:---:|---|
| **«5» (Отлично)** | Выполнены все части выбранного уровня (100% покрытие ТЗ). Запросы оформлены профессионально с плейсхолдерами `?`. Корректно применен `HAVING` (без агрегатов в `WHERE`). В `LEFT JOIN` условия фильтрации правой таблицы не ломают левое соединение. Результаты аккуратно выведены таблицами Pandas, файл CSV успешно экспортирован. |
| **«4» (Хорошо)** | Выполнено 80–90% заданий. Логика запросов верна, но допущены мелкие неточности: пропущен один из агрегатов, забыт `ROUND` при делении, либо пороговые значения захардкожены без плейсхолдеров `?`. Блокнот полностью запускается от начала до конца. |
| **«3» (Удовлетворительно)** | Выполнено 60–75% заданий. Допущены концептуальные ошибки: агрегат написан в `WHERE` вместо `HAVING`, перепутаны `INNER` и `LEFT` JOIN, пропущена фильтрация `IS NULL` при поиске сирот, либо работа оформлена небрежно без разбивки на ячейки. |
| **«2» (Неудовлетворительно)** | Выполнено менее 50% программы лабораторной работы, код содержит фатальные синтаксические ошибки SQL, препятствующие исполнению, либо обнаружен факт несамостоятельного выполнения (прямой плагиат чужого варианта). |
